In [1]:
import os
import numpy as np
import cv2
import torch
from pathlib import Path
from collections import defaultdict
from boxmot import BotSort
from ultralytics import YOLO

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [14]:
model = YOLO(r'weights_sdu_v3\best_sdu_v3.pt')
tracker = BotSort(reid_weights=Path(r'weights_botsort_boxmot\osnet_x0_25_msmt17.pt'), device='cuda:0', half=True)

2025-09-23 10:49:40.385 | INFO     | boxmot.utils.torch_utils:select_device:78 - Yolo Tracking v15.0.1 🚀 Python-3.10.18 torch-2.7.1+cu126
CUDA:0 (NVIDIA GeForce RTX 3060, 12287MiB)
2025-09-23 10:49:40.387 | ERROR    | boxmot.appearance.backends.base_backend:download_model:152 - Found existing ReID weights at weights_botsort_boxmot\osnet_x0_25_msmt17.pt; skipping download.
2025-09-23 10:49:40.671 | SUCCESS  | boxmot.appearance.reid.registry:load_pretrained_weights:64 - Loaded pretrained weights from weights_botsort_boxmot\osnet_x0_25_msmt17.pt


In [11]:
# def detect_yolo(frame):
#     results = model(frame, verbose=False)
#     dets = []
#     for result in results:
#         for box in result.boxes:
#             x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
#             conf = box.conf[0].cpu().numpy()
#             cls = box.cls[0].cpu().numpy()
#             dets.append([x1, y1, x2, y2, conf, cls])
#     return np.array(dets) if dets else np.empty((0, 6))
selected_id = None
current_tracks = []
def detect_yolo(frame):
    detection = []
    theshold_conf = 0.25
    result = model(frame, 
                   imgsz = 1024,
                   conf=0.25, 
                   classes=[0, 1, 2, 3],
                   iou=0.2,
                   device = 'cuda',
                   verbose=False)
    boxes = result[0].boxes.xyxy.cpu()
    confidences = result[0].boxes.conf.cpu()
    idx = [int(i) for i in list(result[0].boxes.cls.cpu())]
    for box, conf, id_cl in zip(boxes, confidences, idx):
        x1 = box[0]
        y1 = box[1]
        x2 = box[2]
        y2 = box[3]
        detection.append([x1, y1, x2, y2, conf, id_cl])
    return np.array(detection, dtype=np.float32)


def mouse_click(event, x, y, flags, param):
    global selected_id, frame

    if event == cv2.EVENT_LBUTTONDOWN:
        print(f'Click to coordinate: ({x}, {y})')

        # Проверяем попал ли клик в какойто bb
        for bbox in current_tracks:
            x1, y1, x2, y2, track_id = bbox[:5]
            if x1 <= x < x2 and y1 <= y <= y2:
                print(f'Choice object: {bbox}')
                selected_id = int(track_id)
                print(f'Choice object with ID: {selected_id}')
                break

In [ ]:
track_history = defaultdict(lambda: [])
path_video = 0 #r'E:\TRACKIING\video\DJIG0103.mp4'
# path_video = r'D:\DB_RECOG_VIDEO_COPTER\source_video\newvideo_07_2025_for_sdu\test_video\korovs_30.mp4'
# path_video = r'D:\DB_RECOG_VIDEO_COPTER\source_video\Artem_video\video_square_Moskow_2025\IMG_2529_.mp4'


# tracker = None
# tracking = False


cap = cv2.VideoCapture(path_video)
# Создаем окно и устанавливаем callback мыши
cv2.namedWindow("Tracking")
cv2.setMouseCallback("Tracking", mouse_click)

frame_id = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    
        
    det = detect_yolo(frame)

    tracks = tracker.update(det, frame)
    
    current_tracks = tracks

    for track in tracks:

        x1, y1, x2, y2, track_id, conf, cls = map(int, track[:7])
        color = (0, 0, 255)
        if selected_id is not None and track_id == selected_id:
            color_ = (255, 0, 0)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color_, 2)
            # print(track_id)
            cv2.putText(frame, f'Selected {track_id}', (x1, y1 - 10), cv2.FONT_HERSHEY_SCRIPT_SIMPLEX, 0.75, color_, 2)

        else:
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 1)
            cv2.putText(frame, f"ID:{track_id}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)


        
    cv2.imshow('Tracking', frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('r'):
        selected_id = None
        print("selected drop...")
        
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    
    frame_id += 1
cap.release()
cv2.destroyAllWindows()